## 面试问题

Scratchpad/working memory 在循环内怎么维护与失效？

## 回答主线

scratchpad 存推测（假设/计划），history 存事实。假设要带来源与置信度，新观察推翻旧假设时应失效更新而非叠加。本 Notebook 让 agent 先低置信假设「意图=退款」，再收到更权威的「换货」，对比叠加式 scratchpad（保留矛盾假设、错误退款）与失效式 scratchpad（更新为换货、正确）。

## 真实案例

客服 agent 先假设意图=退款(置信0.5)，随后用户明确说要换尺码(置信0.9)。对比是否用新观察失效旧假设。数据为教学事件，不代表真实客服系统。

In [1]:
observations = [  # 定义一段会推翻早期假设的观察序列。
    {"step": 1, "text": "我要退款", "field": "intent", "value": "refund", "confidence": 0.5},  # 早期低置信意图。
    {"step": 2, "text": "其实我想换一个尺码", "field": "intent", "value": "exchange", "confidence": 0.9},  # 后续更权威的意图。
]  # 结束观察序列定义。

print("观察步数:", len(observations))  # 展示观察数量。
for ob in observations:  # 逐条打印观察。
    print("  ", ob["text"], "-> intent", ob["value"], "conf", ob["confidence"])  # 展示每步意图与置信度。

观察步数: 2
   我要退款 -> intent refund conf 0.5
   其实我想换一个尺码 -> intent exchange conf 0.9


## 基线（Baseline）

反面基线：append-only scratchpad，每个观察都新增一条假设，从不失效。结果同一个 `intent` 字段并存 refund 和 exchange 两个矛盾假设。

In [2]:
def append_only_scratchpad(observations):  # 无失效的 scratchpad：假设只叠加不更新。
    notes = []  # 收集假设条目。
    for ob in observations:  # 遍历观察。
        notes.append({"field": ob["field"], "value": ob["value"]})  # 每个观察都新增一条假设。
    return notes  # 返回叠加后的假设列表。

naive_notes = append_only_scratchpad(observations)  # 生成叠加式 scratchpad。
print("叠加式 scratchpad:", naive_notes)  # 展示同时保留两个矛盾假设。
print("intent 假设数:", sum(1 for n in naive_notes if n["field"] == "intent"))  # 展示矛盾假设并存。

叠加式 scratchpad: [{'field': 'intent', 'value': 'refund'}, {'field': 'intent', 'value': 'exchange'}]
intent 假设数: 2


## 失败案例与修正

叠加式若取到最早的假设就会错误退款。修正是失效式 scratchpad：按字段索引，收到同字段更权威观察时更新旧假设，只保留当前最可信的理解。

In [3]:
def maintained_scratchpad(observations):  # 带失效的 scratchpad：同字段新观察更新旧假设。
    notes = {}  # 用字段名索引当前假设。
    for ob in observations:  # 遍历观察。
        current = notes.get(ob["field"])  # 取该字段现有假设。
        if current is None or ob["confidence"] >= current["confidence"]:  # 更权威时失效旧假设。
            notes[ob["field"]] = {"value": ob["value"], "confidence": ob["confidence"]}  # 更新为新假设。
    return notes  # 返回失效更新后的假设。

maintained = maintained_scratchpad(observations)  # 生成带失效的 scratchpad。
print("失效更新后 scratchpad:", maintained)  # 展示 intent 被更新为 exchange。
print("当前 intent 假设:", maintained["intent"]["value"])  # 展示最新意图。

失效更新后 scratchpad: {'intent': {'value': 'exchange', 'confidence': 0.9}}
当前 intent 假设: exchange


In [4]:
def decide_action(intent):  # 依据当前意图决定动作。
    if intent == "refund":  # 意图为退款。
        return "issue_refund"  # 发起退款。
    if intent == "exchange":  # 意图为换货。
        return "start_exchange"  # 发起换货。
    return "ask_user"  # 意图不明则追问。

naive_intent = naive_notes[0]["value"]  # 叠加式可能取到最早的假设。
naive_action = decide_action(naive_intent)  # 基于过期假设的决策。
maintained_action = decide_action(maintained["intent"]["value"])  # 基于失效更新后的决策。
print("叠加式(取首个假设)动作:", naive_action, "(基于过期的 refund)")  # 展示基于过期假设错误退款。
print("失效更新后动作:", maintained_action, "(基于最新的 exchange)")  # 展示正确换货。

叠加式(取首个假设)动作: issue_refund (基于过期的 refund)
失效更新后动作: start_exchange (基于最新的 exchange)


## 结果解读

叠加式 scratchpad 保留两个矛盾假设、按最早的 refund 错误决策；失效式按置信度更新，intent 变为 exchange，正确换货。要点：假设带置信度与来源，冲突时失效而非叠加，且推测不能自动升格为事实。

In [5]:
print("叠加式 intent 假设数:", sum(1 for n in naive_notes if n["field"] == "intent"), "(矛盾并存)")  # 叠加式保留矛盾。
print("失效式 intent 假设数: 1 (唯一最新)")  # 失效式只保留最新。
print("动作差异:", naive_action, "vs", maintained_action)  # 展示两者决策不同。

叠加式 intent 假设数: 2 (矛盾并存)
失效式 intent 假设数: 1 (唯一最新)
动作差异: issue_refund vs start_exchange


In [6]:
assert sum(1 for n in naive_notes if n["field"] == "intent") == 2  # 叠加式保留两个矛盾假设。
assert maintained["intent"]["value"] == "exchange"  # 失效式更新为最新意图。
assert naive_action == "issue_refund"  # 叠加式基于过期假设错误退款。
assert maintained_action == "start_exchange"  # 失效式基于最新假设正确换货。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
